# Policy Agent 실험

연금 가입, 이전, 해지와 제도상 가능 여부를 검색하고 최종 `DomainResult`로 제출하는 Policy Agent의 전체 흐름을 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 연금 제도 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

Agent와 외부 연결은 한 번만 만들어 재사용한다. 커밋 전에는 검색 원문과 판단 결과 출력을 모두 지운다.

In [ ]:
from pprint import pprint

# Policy Agent와 HCX 모델
from pension_agent.agent.model_factory import create_chat_clovax
from pension_agent.agent.policy import create_policy_agent
from pension_agent.agent.search import SearchService

# CLOVA와 Qdrant 연결 설정
from pension_agent.config import (
    BGE_M3_EMBEDDING_CONFIG,
    DEFAULT_DOMAIN_AGENT_HCX_CONFIG,
    ClovaStudioConnection,
    QdrantConnection,
)
from pension_agent.retrieval import (
    create_async_qdrant_client,
    create_async_qdrant_retriever,
    create_clova_query_embedder,
    prewarm_kiwi,
)

## Policy Agent 조립

`.env`의 설정으로 HCX와 Search Service를 준비해 Policy Agent에 연결한다.

In [ ]:
# 셀을 다시 실행하면 기존 Qdrant 연결을 먼저 닫는다.
existing_qdrant_client = globals().get("qdrant_client")
if existing_qdrant_client is not None:
    await existing_qdrant_client.close()

clova_connection = ClovaStudioConnection()
qdrant_connection = QdrantConnection()
model = create_chat_clovax(
    config=DEFAULT_DOMAIN_AGENT_HCX_CONFIG,
    connection=clova_connection,
)
embedder = create_clova_query_embedder(
    config=BGE_M3_EMBEDDING_CONFIG,
    connection=clova_connection,
)
await prewarm_kiwi()
qdrant_client = create_async_qdrant_client(qdrant_connection)
retriever = create_async_qdrant_retriever(
    qdrant_client,
    connection=qdrant_connection,
)
search_service = SearchService(embedder=embedder, retriever=retriever)
policy_agent = create_policy_agent(model=model, search_service=search_service)

# API key를 제외한 실행 대상만 확인한다.
{
    "qdrant_url": qdrant_connection.url,
    "collection": qdrant_connection.collection,
    "hcx_model": DEFAULT_DOMAIN_AGENT_HCX_CONFIG.model,
    "embedding_model": BGE_M3_EMBEDDING_CONFIG.model,
}

## 질문 실행 함수

질문 원문과 Policy Agent가 수행할 하나의 판단 목표를 전달한다.

In [ ]:
async def ask_policy_agent(*, question: str, objective: str) -> dict[str, object]:
    # Policy Agent를 직접 호출해 검증된 DomainResult를 받는다.
    return await policy_agent(
        {
            "question": question,
            "objective": objective,
        }
    )

## 계좌 이전 절차 판단

이전 가능 여부와 절차를 판단하고 결론에 실제 사용한 제도 문서 근거를 확인한다.

In [ ]:
transfer_result = await ask_policy_agent(
    question="IRP 계좌를 다른 금융회사로 이전하려면 어떤 조건과 절차를 확인해야 해?",
    objective="IRP 계좌의 금융회사 간 이전 가능 여부와 필요한 절차 판단",
)
pprint(transfer_result)

In [ ]:
# 결론에 선택된 근거 문서와 위치만 빠르게 확인한다.
[(evidence["source_file_name"], evidence["locator"]) for evidence in transfer_result["evidence"]]

## 누락 조건이 있는 해지 판단

가입자와 계약 조건이 충분하지 않은 질문에서 단정하지 않고 필요한 조건을 반환하는지 확인한다.

In [ ]:
conditional_result = await ask_policy_agent(
    question="연금저축을 지금 해지할 수 있나요?",
    objective="연금저축 해지 가능 여부와 판단에 필요한 누락 조건 확인",
)
pprint(conditional_result["decision"])
pprint(conditional_result["warnings"])

## 정리

실험을 마치면 Qdrant 연결을 닫는다. 다시 질문하려면 Policy Agent 조립부터 실행한다.

In [ ]:
# 외부 연결을 정리한다.
await qdrant_client.close()
qdrant_client = None
"Policy Agent 연결 정리 완료"